In [1]:
import torch 
import numpy as np
import sentence_transformers
import datasets
import transformers

In [2]:
ds = datasets.load_dataset("iitolstykh/LLMTrace_classification")

README.md: 0.00B [00:00, ?B/s]

train.jsonl:   0%|          | 0.00/1.40G [00:00<?, ?B/s]

valid.jsonl:   0%|          | 0.00/292M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/318M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/411440 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/86696 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/90950 [00:00<?, ? examples/s]

In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['lang', 'label', 'model', 'data_type', 'prompt_type', 'topic_id', 'text', 'prompt'],
        num_rows: 411440
    })
    validation: Dataset({
        features: ['lang', 'label', 'model', 'data_type', 'prompt_type', 'topic_id', 'text', 'prompt'],
        num_rows: 86696
    })
    test: Dataset({
        features: ['lang', 'label', 'model', 'data_type', 'prompt_type', 'topic_id', 'text', 'prompt'],
        num_rows: 90950
    })
})

In [4]:
texts_train = list(ds['train']['text'])
texts_valid = list(ds['validation']['text'])
texts_test = list(ds['test']['text'])
y_train = np.array([0 if label == 'ai' else 1 for label in ds['train']['label']])
y_valid = np.array([0 if label == 'ai' else 1 for label in ds['validation']['label']])
y_test = np.array([0 if label == 'ai' else 1 for label in ds['test']['label']])
print(f'len texts_train: {len(texts_train)}, len y_train: {len(y_train)}')
print(f'len texts_valid: {len(texts_valid)}, len y_valid: {len(y_valid)}')
print(f'len texts_test: {len(texts_test)}, len y_test: {len(y_test)}')

len texts_train: 411440, len y_train: 411440
len texts_valid: 86696, len y_valid: 86696
len texts_test: 90950, len y_test: 90950


In [5]:
type(texts_train)

list

In [6]:
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = sentence_transformers.SentenceTransformer(model_name_or_path='intfloat/multilingual-e5-base', 
                                                  # device=device
                                                  )
model.half()
# print(device)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [7]:
print(f'max seq len for model: {model.max_seq_length}')

max seq len for model: 512


In [8]:
# Проверка, доступна ли CUDA вообще
if torch.cuda.is_available():
    # 1. Узнать количество доступных GPU
    count = torch.cuda.device_count()
    print(f"Доступно CUDA-устройств: {count}")

    # 2. Вывести список всех устройств
    for i in range(count):
        print(f"Устройство {i}: {torch.cuda.get_device_name(i)}")

Доступно CUDA-устройств: 2
Устройство 0: Tesla T4
Устройство 1: Tesla T4


In [9]:
pool = model.start_multi_process_pool()

embeddings_train = model.encode(texts_train, 
                                prompt='query: ', 
                                batch_size=256, 
                                convert_to_numpy=True, 
                                # output_value='sentence_embeddings',
                                # device=device,
                                # normalize_embeddings=True,
                                show_progress_bar = True, 
                                pool=pool, 
                               )

model.stop_multi_process_pool(pool)

Chunks:   0%|          | 0/83 [00:00<?, ?it/s]

In [10]:
embeddings_train.shape

(411440, 768)

In [11]:
np.save('embeddings_train.npy', embeddings_train)

In [13]:
pool = model.start_multi_process_pool()

embeddings_valid = model.encode(texts_valid, 
                                prompt='query: ', 
                                batch_size=256, 
                                convert_to_numpy=True, 
                                # output_value='sentence_embeddings',
                                # device=device,
                                # normalize_embeddings=True,
                                show_progress_bar = True, 
                                pool=pool, 
                               )

model.stop_multi_process_pool(pool)

Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

In [14]:
np.save('embeddings_valid.npy', embeddings_valid)

In [15]:
pool = model.start_multi_process_pool()

embeddings_test = model.encode(texts_test, 
                                prompt='query: ', 
                                batch_size=256, 
                                convert_to_numpy=True, 
                                # output_value='sentence_embeddings',
                                # device=device,
                                # normalize_embeddings=True,
                                show_progress_bar = True, 
                                pool=pool, 
                               )

model.stop_multi_process_pool(pool)

Chunks:   0%|          | 0/20 [00:00<?, ?it/s]

In [16]:
np.save('embeddings_test.npy', embeddings_test)